In [7]:
import torch
from torchvision.datasets import OxfordIIITPet
from torchvision import transforms
from torch.utils.data import DataLoader
import wandb

In [8]:
import torchvision.models as models
from torchvision.models import ResNet50_Weights

weights = ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)
transform = weights.transforms()


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\shehr/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:21<00:00, 4.76MB/s]


In [9]:
dataset = OxfordIIITPet(
    root="data",
    split="trainval",
    target_types="category",
    transform=transform,
    download=True,
)

loader = DataLoader(dataset, batch_size=32, shuffle=True)

images, labels = next(iter(loader))
print(images.shape)   # [32, 3, 224, 224]
print(labels.shape)   # [32]

torch.Size([32, 3, 224, 224])
torch.Size([32])


In [10]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

Model 2: Full fine-tuning at low LR

In [21]:
run = wandb.init(
    # Set the wandb entity where your project will be logged (generally your team name).
    entity="shehryar11w-private",
    # Set the wandb project where this run will be logged.
    project="petDataset",
    # Track hyperparameters and run metadata.
    config={
        "learning_rate": 1e-3,
        "architecture": "ResNet - Full Finetuning",
        "dataset": "OxfordIIITPet",
        "epochs": 15,
    },
)


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_loss,█▃▂▂▂▁▁▁▁▁▁▁▁▁▁
val_accuracy,▁▅▆▆▇▆█▇▇▇█▇▇█▇
val_loss,█▄▃▂▂▂▂▁▁▁▁▁▁▁▁
epoch,15
train_loss,0.04881
val_accuracy,92.11957
val_loss,0.24224


wandb: Initializing weave.


In [22]:
import torch.nn as nn
import torch.nn.functional as F

weights = ResNet50_Weights.DEFAULT
model = models.resnet50(weights=weights)
transform = weights.transforms()


model.fc = nn.Linear(model.fc.in_features, 37)

In [23]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [24]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(),lr=1e-5)

In [25]:
for epoch in range(15):  # loop over the dataset multiple times

    ### Train
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)
        # zero the parameter gradients
        optimizer.zero_grad()

        # forward + backward + optimize
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        # log statistics
        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    print(f'[{epoch + 1}] loss: {epoch_loss:.4f}')

    ### Validate
    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in val_loader:

            inputs = inputs.to(device)
            labels = labels.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

    val_loss /= len(val_loader)
    val_acc = 100 * correct / total

    print(f"Epoch {epoch+1}: Val Loss = {val_loss:.4f}, Val Acc = {val_acc:.2f}%")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": epoch_loss,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
    })

print('Finished Training')

[1] loss: 3.5477
Epoch 1: Val Loss = 3.4601, Val Acc = 16.58%
[2] loss: 3.2701
Epoch 2: Val Loss = 3.0849, Val Acc = 47.42%
[3] loss: 2.7503
Epoch 3: Val Loss = 2.4287, Val Acc = 73.37%
[4] loss: 2.0622
Epoch 4: Val Loss = 1.7313, Val Acc = 82.47%
[5] loss: 1.4435
Epoch 5: Val Loss = 1.2227, Val Acc = 87.36%
[6] loss: 1.0020
Epoch 6: Val Loss = 0.8696, Val Acc = 90.49%
[7] loss: 0.7022
Epoch 7: Val Loss = 0.6477, Val Acc = 91.30%
[8] loss: 0.5177
Epoch 8: Val Loss = 0.4917, Val Acc = 91.85%
[9] loss: 0.3921
Epoch 9: Val Loss = 0.4138, Val Acc = 93.34%
[10] loss: 0.2919
Epoch 10: Val Loss = 0.3649, Val Acc = 93.89%
[11] loss: 0.2349
Epoch 11: Val Loss = 0.3202, Val Acc = 93.48%
[12] loss: 0.1805
Epoch 12: Val Loss = 0.2993, Val Acc = 93.75%
[13] loss: 0.1483
Epoch 13: Val Loss = 0.2863, Val Acc = 93.48%
[14] loss: 0.1218
Epoch 14: Val Loss = 0.2685, Val Acc = 93.75%
[15] loss: 0.1021
Epoch 15: Val Loss = 0.2571, Val Acc = 94.02%
Finished Training


In [26]:
wandb.finish()

wandb: ERROR The nbformat package was not found. It is required to save notebook history.


epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_loss,█▇▆▅▄▃▂▂▂▁▁▁▁▁▁
val_accuracy,▁▄▆▇▇██████████
val_loss,█▇▆▄▃▂▂▂▁▁▁▁▁▁▁
epoch,15
train_loss,0.10211
val_accuracy,94.02174
val_loss,0.25711
